In [ ]:
import hashlib
import os
import multiprocessing as mp
from ecdsa import SigningKey, SECP256k1
from time import time
from collections import defaultdict
import random
import json
import logging

# Set up logging
logging.basicConfig(filename='debug_log.txt', level=logging.DEBUG,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# secp256k1 curve order
n = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141

# Function to compute Hamming distance
def hamming_distance(bytes1, bytes2):
    if len(bytes1) != len(bytes2):
        raise ValueError("Byte strings must have equal length")
    return sum(bin(b1 ^ b2).count('1') for b1, b2 in zip(bytes1, bytes2))

# Function to derive public key and coordinates
def get_public_key(private_key_bytes, compressed=False):
    try:
        sk = SigningKey.from_string(private_key_bytes, curve=SECP256k1)
        vk = sk.verifying_key
        pubkey_bytes = vk.to_string("compressed" if compressed else "uncompressed")
        x = int.from_bytes(pubkey_bytes[1:33], 'big')
        y = int.from_bytes(pubkey_bytes[33:65], 'big') if not compressed else None
        return pubkey_bytes, x, y
    except Exception as e:
        logging.error(f"Error in get_public_key: {e}")
        raise

# Target public key and coordinates
target_public_key = bytes.fromhex(
    "04d6597d465408e6e11264c116dd98b539740e802dc756d7eb88741696e20dfe7d3588695d2e7ad23cbf0aa056d42afada63036d66a1d9b97070dd6bc0c87ceb0d"
)
target_x = int.from_bytes(target_public_key[1:33], 'big')
target_y = int.from_bytes(target_public_key[33:65], 'big')

# Simulate r value for signature
def simulate_r_value(private_key_bytes):
    try:
        nonce = os.urandom(32)
        pubkey, _, _ = get_public_key(nonce, compressed=False)
        r = int.from_bytes(pubkey[1:33], 'big')
        return r
    except Exception as e:
        logging.error(f"Error in simulate_r_value: {e}")
        raise

# Check if r value difference has leading digit 7 or 8
def has_78_leading_digit(diff):
    try:
        diff_str = f"{diff:.2e}"
        leading_digit = diff_str[0]
        return leading_digit in ('7', '8')
    except Exception as e:
        logging.error(f"Error in has_78_leading_digit: {e}")
        return False

# Pollard’s Kangaroo algorithm
def pollard_kangaroo(private_key_bytes, range_bounds, max_steps=200000, dp_bits=12):
    try:
        k = int.from_bytes(private_key_bytes, 'big')
        L, U = range_bounds
        if L < 1 or U >= n or U - L < 2 or U - L > 2**32:
            L = max(1, k - 2**18)
            U = min(n - 1, k + 2**18)

        w = U - L
        steps = int(2 * (w ** 0.5))
        dp_mask = (1 << dp_bits) - 1
        tame_points = {}
        wild_points = {}
        jump_sizes = [2**i for i in range(16)]

        k_tame = random.randint(L, U)
        point_tame, _, _ = get_public_key(k_tame.to_bytes(32, 'big'), compressed=False)
        distance_tame = 0
        for _ in range(max_steps):
            x = int.from_bytes(point_tame[1:33], 'big')
            jump_idx = x % len(jump_sizes)
            distance_tame += jump_sizes[jump_idx]
            k_tame = (k_tame + jump_sizes[jump_idx]) % n
            point_tame, _, _ = get_public_key(k_tame.to_bytes(32, 'big'), compressed=False)
            if int.from_bytes(point_tame, 'big') & dp_mask == 0:
                tame_points[point_tame] = (k_tame, distance_tame)

            if point_tame in wild_points:
                k_wild, distance_wild = wild_points[point_tame]
                k_candidate = (k_tame - distance_tame + distance_wild) % n
                pubkey, x_cand, y_cand = get_public_key(k_candidate.to_bytes(32, 'big'), compressed=False)
                distance = hamming_distance(pubkey, target_public_key)
                if distance == 0:
                    print(f"Private key found: {k_candidate}, Distance: 0")
                    return k_candidate.to_bytes(32, 'big'), 0
                if x_cand < target_x and y_cand > target_y:
                    return k_candidate.to_bytes(32, 'big'), distance

            if len(tame_points) > steps:
                break

        k_wild = k
        point_wild, x_wild, y_wild = get_public_key(k_wild.to_bytes(32, 'big'), compressed=False)
        distance_wild = 0
        best_distance = float('inf')
        best_k = k_wild
        for _ in range(max_steps):
            x = int.from_bytes(point_wild[1:33], 'big')
            jump_idx = x % len(jump_sizes)
            distance_wild += jump_sizes[jump_idx]
            k_wild = (k_wild + jump_sizes[jump_idx]) % n
            point_wild, x_wild, y_wild = get_public_key(k_wild.to_bytes(32, 'big'), compressed=False)
            if int.from_bytes(point_wild, 'big') & dp_mask == 0:
                wild_points[point_wild] = (k_wild, distance_wild)

            if point_wild in tame_points:
                k_tame, distance_tame = tame_points[point_wild]
                k_candidate = (k_tame - distance_tame + distance_wild) % n
                pubkey, x_cand, y_cand = get_public_key(k_candidate.to_bytes(32, 'big'), compressed=False)
                distance = hamming_distance(pubkey, target_public_key)
                if distance == 0:
                    print(f"Private key found: {k_candidate}, Distance: 0")
                    return k_candidate.to_bytes(32, 'big'), 0
                if distance < best_distance and x_cand < target_x and y_cand > target_y:
                    best_distance = distance
                    best_k = k_candidate

            if len(wild_points) > steps:
                break

        return best_k.to_bytes(32, 'big'), best_distance
    except Exception as e:
        logging.error(f"Error in pollard_kangaroo: {e}")
        raise

# Hill-climbing with fine steps
def hill_climb(private_key_bytes, max_iterations=5000, deltas=[1, 2], perturb_range=10000):
    try:
        best_k = int.from_bytes(private_key_bytes, 'big')
        best_distance = float('inf')
        best_x, best_y = None, None
        range_bounds_x = [best_k, best_k]
        range_bounds_y = [best_k, best_k]
        events = []
        steps = []

        for i in range(max_iterations):
            adjustments = []
            for delta in deltas:
                adjustments.extend([
                    (best_k + delta, f"inc_{delta}_{i+1}"),
                    (best_k - delta, f"dec_{delta}_{i+1}")
                ])
            adjustments.extend([
                ((best_k * 2) % n, "scale_2"),
                ((best_k * 3) % n, "scale_3"),
                ((best_k * 4) % n, "scale_4"),
                (best_k + random.randint(-perturb_range, perturb_range), f"perturb_{i}")
            ])

            for k_new, step_type in adjustments:
                if k_new < 1 or k_new >= n:
                    continue
                k_bytes = k_new.to_bytes(32, 'big')
                pubkey, x_new, y_new = get_public_key(k_bytes, compressed=False)
                distance = hamming_distance(pubkey, target_public_key)

                if x_new == target_x:
                    events.append((k_new, "x_exact_match", distance, x_new, y_new))
                if y_new == target_y:
                    events.append((k_new, "y_exact_match", distance, x_new, y_new))
                if distance == 0:
                    print(f"Private key found: {k_new}, Distance: 0")
                    return k_bytes, 0, [k_new, k_new], events, steps

                if best_x is not None:
                    if x_new > target_x and best_x < target_x:
                        events.append((k_new, "x_cross_to_above", distance, x_new, y_new))
                        range_bounds_x = [min(range_bounds_x[0], min(best_k, k_new)), max(range_bounds_x[1], max(best_k, k_new))]
                    elif x_new < target_x and best_x > target_x:
                        events.append((k_new, "x_cross_to_below", distance, x_new, y_new))
                        range_bounds_x = [min(range_bounds_x[0], min(best_k, k_new)), max(range_bounds_x[1], max(best_k, k_new))]
                if best_y is not None:
                    if y_new < target_y and best_y > target_y:
                        events.append((k_new, "y_cross_to_below", distance, x_new, y_new))
                        range_bounds_y = [min(range_bounds_y[0], min(best_k, k_new)), max(range_bounds_y[1], max(best_k, k_new))]
                    elif y_new > target_y and best_y < target_y:
                        events.append((k_new, "y_cross_to_above", distance, x_new, y_new))
                        range_bounds_y = [min(range_bounds_y[0], min(best_k, k_new)), max(range_bounds_y[1], max(best_k, k_new))]

                if (distance < best_distance) or (x_new < target_x and y_new > target_y and distance <= best_distance + 2):
                    best_distance = distance
                    best_k = k_new
                    best_x, best_y = x_new, y_new
                    steps.append((step_type, distance, x_new < target_x, y_new > target_y))

            range_bounds = [max(range_bounds_x[0], range_bounds_y[0]), min(range_bounds_y[1], range_bounds_x[1], best_k + 2**26)]
            range_bounds[0] = max(1, best_k - 2**26)
            range_bounds[1] = min(n - 1, best_k + 2**26)

            if range_bounds[1] - range_bounds[0] < 2**26:
                break

        return best_k.to_bytes(32, 'big'), best_distance, range_bounds, events, steps
    except Exception as e:
        logging.error(f"Error in hill_climb: {e}")
        raise

# Brute-force test for Weak RNG seeds with checkpointing
def test_seed_range(seed_range, r_values_ref, checkpoint_file, checkpoint_interval=10000000):
    try:
        start_seed, end_seed = seed_range
        candidates = []
        distances = []
        x_below_count = 0
        y_above_count = 0

        # Load checkpoint if exists
        checkpoint_data = {"last_seed": start_seed - 1, "candidates": []}
        if os.path.exists(checkpoint_file):
            with open(checkpoint_file, 'r') as f:
                checkpoint_data = json.load(f)
            start_seed = max(start_seed, checkpoint_data["last_seed"] + 1)
            candidates = checkpoint_data["candidates"]

        for seed in range(start_seed, end_seed):
            private_key_bytes = hashlib.sha256(seed.to_bytes(4, 'big')).digest()
            pubkey, x, y = get_public_key(private_key_bytes, compressed=False)
            distance = hamming_distance(pubkey, target_public_key)

            if x == target_x:
                candidates.append({"seed": seed, "event": "x_exact_match", "distance": distance})
            if y == target_y:
                candidates.append({"seed": seed, "event": "y_exact_match", "distance": distance})
            if distance == 0:
                print(f"Private key found: Seed {seed}, Distance: 0")
                return private_key_bytes, 0, candidates, distances, x_below_count, y_above_count

            if x < target_x and y > target_y:
                distances.append(distance)
                x_below_count += 1
                y_above_count += 1
                r_value = simulate_r_value(private_key_bytes)
                for r_ref in r_values_ref:
                    diff = abs(r_value - r_ref)
                    if has_78_leading_digit(diff):
                        candidates.append({
                            "seed": seed,
                            "private_key": private_key_bytes.hex(),
                            "distance": distance,
                            "x": x,
                            "y": y,
                            "x_below": x < target_x,
                            "y_above": y > target_y,
                            "x_match": x == target_x,
                            "y_match": y == target_y,
                            "r_value": r_value
                        })
                        candidates.sort(key=lambda x: x["distance"])
                        candidates = candidates[:200]
            else:
                distances.append(distance)

            # Checkpoint every 10M seeds
            if (seed + 1) % checkpoint_interval == 0:
                checkpoint_data = {
                    "last_seed": seed,
                    "candidates": candidates,
                    "distances": distances,
                    "x_below_count": x_below_count,
                    "y_above_count": y_above_count
                }
                with open(checkpoint_file, 'w') as f:
                    json.dump(checkpoint_data, f)
                print(f"Checkpoint saved at seed {seed}, {len(distances)} seeds processed...")

        return None, float('inf'), candidates, distances, x_below_count, y_above_count
    except Exception as e:
        logging.error(f"Error in test_seed_range: {e}")
        raise

# Test other key methods
def test_key(args):
    try:
        private_key_bytes, index = args
        logging.debug(f"Processing key with index {index}")
        pubkey, x, y = get_public_key(private_key_bytes, compressed=False)
        distance = hamming_distance(pubkey, target_public_key)
        r_value = simulate_r_value(private_key_bytes)
        return {
            "index": index,
            "private_key": private_key_bytes.hex(),
            "distance": distance,
            "x": x,
            "y": y,
            "x_below": x < target_x,
            "y_above": y > target_y,
            "x_match": x == target_x,
            "y_match": y == target_y,
            "r_value": r_value
        }
    except Exception as e:
        logging.error(f"Error in test_key with index {index}: {e}")
        raise

# Fixed key generators to avoid negative indices/values
def weak_rng_keys(n):
    try:
        centers = [6670011, 5340566, 210434]
        for i in range(n):
            offset = i - (n // 6)  # ±0.33M
            seed = centers[i % len(centers)] + offset
            # Ensure non-negative seed and index
            seed = max(0, seed) % (2**32)  # Clamp to 32-bit unsigned range
            index = i % (2**31)  # Ensure positive index
            logging.debug(f"Generating Weak RNG seed: {seed}, index: {index}")
            yield (hashlib.sha256(seed.to_bytes(4, 'big')).digest(), index)
    except Exception as e:
        logging.error(f"Error in weak_rng_keys: {e}")
        raise

def brain_wallet_keys(n):
    try:
        base_phrases = [
            "bitcoin", "satoshi", "blockchain", "crypto", "2010", "password", "mining",
            "block", "9000", "072410", "satoshi2010", "block9000", "mining2010", "bitcoin9000",
            "satoshi072410", "block072410", "wallet967", "satoshi9000", "bitcoin072410",
            "mining9000", "wallet9672010", "satoshi967", "block967", "mining967", "bitcoin967"
        ]
        for i in range(n):
            phrase = f"{base_phrases[i % len(base_phrases)]}{(i % 1000000)}"
            index = i % (2**31)  # Ensure positive index
            logging.debug(f"Generating Brain Wallet phrase: {phrase}, index: {index}")
            yield (hashlib.sha256(phrase.encode()).digest(), index)
    except Exception as e:
        logging.error(f"Error in brain_wallet_keys: {e}")
        raise

def byte_pair_keys(n):
    try:
        fixed_bytes = [bytes.fromhex("0000"), bytes.fromhex("0704"), bytes.fromhex("2010"),
                       bytes.fromhex("9000"), bytes.fromhex("0724"), bytes.fromhex("0710"),
                       bytes.fromhex("9670"), bytes.fromhex("0720"), bytes.fromhex("0714"),
                       bytes.fromhex("9671")]
        for i in range(min(n, 2**16 * len(fixed_bytes))):
            base = fixed_bytes[i // (2**16)]
            pair = (i % (2**16)).to_bytes(2, 'big')
            index = i % (2**31)  # Ensure positive index
            logging.debug(f"Generating Byte Pair index: {index}")
            yield (base + pair + os.urandom(28), index)
    except Exception as e:
        logging.error(f"Error in byte_pair_keys: {e}")
        raise

# Main script
if __name__ == "__main__":
    print("Starting optimized private key search with CPU brute-force and checkpointing...")

    # Top candidates
    top_candidates = {
        "Weak RNG": [
            (hashlib.sha256((6670011).to_bytes(4, 'big')).digest(), 6670011, 195),
            (hashlib.sha256((5340566).to_bytes(4, 'big')).digest(), 5340566, 195),
            (hashlib.sha256((210434).to_bytes(4, 'big')).digest(), 210434, 195),
        ],
        "Brain Wallet": [
            (hashlib.sha256(f"bitcoin37157".encode()).digest(), 37157, 206),
            (hashlib.sha256(f"satoshi137157".encode()).digest(), 137157, 206),
        ],
        "Byte Pair": [
            (bytes.fromhex("0000" + format(71447 % (2**16), '04x') + os.urandom(28).hex()), 71447, 203),
            (bytes.fromhex("0704" + format(210434 % (2**16), '04x') + os.urandom(28).hex()), 210434, 203),
        ]
    }

    # Hill-climbing for candidates
    print("\nHill-climbing for candidates...")
    refined_candidates = defaultdict(list)
    for method_name, candidates in top_candidates.items():
        for priv_key, idx, orig_distance in candidates:
            try:
                new_priv_key, new_distance, range_bounds, events, steps = hill_climb(priv_key)
                print(f"{method_name} Index {idx}: Original Distance: {orig_distance}, "
                      f"New Distance: {new_distance}, Range: [{range_bounds[0]}, {range_bounds[1]}], "
                      f"Events: {events[:5]}, Steps: {steps[:5]}")
                refined_candidates[method_name].append((new_priv_key, idx, new_distance, range_bounds, events))
            except Exception as e:
                logging.error(f"Error in hill_climb for {method_name} index {idx}: {e}")

    # Kangaroo search on refined ranges
    print("\nPollard’s Kangaroo search on refined ranges...")
    kangaroo_results = defaultdict(list)
    for method_name, candidates in refined_candidates.items():
        for priv_key, idx, distance, range_bounds, events in candidates:
            try:
                kangaroo_key, kangaroo_distance = pollard_kangaroo(priv_key, range_bounds)
                print(f"{method_name} Index {idx}: Kangaroo Distance: {kangaroo_distance}")
                kangaroo_results[method_name].append((kangaroo_key, idx, kangaroo_distance, events))
                if kangaroo_distance == 0:
                    print("Key found by Kangaroo, exiting...")
                    exit(0)
            except Exception as e:
                logging.error(f"Error in pollard_kangaroo for {method_name} index {idx}: {e}")

    # Test Brain Wallet and Byte Pair
    print("\nTesting Brain Wallet and Byte Pair keys...")
    n_keys = {"Brain Wallet": 1000000, "Byte Pair": 500000}
    results = {}
    top_new_candidates = defaultdict(list)
    r_values_ref = [3911235232491858960683741385178800875840269669132487145437180544934203389296]

    with mp.Pool(mp.cpu_count()) as pool:
        for method_name, key_generator in [("Brain Wallet", brain_wallet_keys), ("Byte Pair", byte_pair_keys)]:
            start_time = time()
            distances = []
            x_below_count = 0
            y_above_count = 0
            candidates = []

            try:
                key_iter = key_generator(n_keys[method_name])
                results_iter = pool.imap(test_key, key_iter, chunksize=1000)
                for result in results_iter:
                    if result["x_match"] or result["y_match"]:
                        print(f"{method_name} Index {result['index']}: x_match={result['x_match']}, y_match={result['y_match']}, Distance: {result['distance']}")
                    if result["x_below"] and result["y_above"]:
                        distances.append(result["distance"])
                        x_below_count += 1
                        y_above_count += 1
                        for r_ref in r_values_ref:
                            diff = abs(result["r_value"] - r_ref)
                            if has_78_leading_digit(diff):
                                candidates.append(result)
                                if len(top_new_candidates[method_name]) < 200:
                                    top_new_candidates[method_name].append(result)
                                    top_new_candidates[method_name].sort(key=lambda x: x["distance"])
                                elif result["distance"] < top_new_candidates[method_name][-1]["distance"]:
                                    top_new_candidates[method_name][-1] = result
                                    top_new_candidates[method_name].sort(key=lambda x: x["distance"])
                    else:
                        distances.append(result["distance"])

                min_distance = min(distances) if distances else float('inf')
                mean_distance = sum(distances) / len(distances) if distances else float('inf')
                x_below_ratio = x_below_count / n_keys[method_name]
                y_above_ratio = y_above_count / n_keys[method_name]

                results[method_name] = {
                    "min": min_distance,
                    "mean": mean_distance,
                    "x_below_ratio": x_below_ratio,
                    "y_above_ratio": y_above_ratio,
                    "distances": distances
                }

                print(f"\nMethod: {method_name}")
                print(f"Minimum Hamming Distance: {min_distance}")
                print(f"Mean Hamming Distance: {mean_distance:.2f}")
                print(f"Ratio x < target_x: {x_below_ratio:.3f}")
                print(f"Ratio y > target_y: {y_above_ratio:.3f}")
                print(f"Time taken: {time() - start_time:.2f} seconds")
            except Exception as e:
                logging.error(f"Error in testing {method_name} keys: {e}")
                print(f"Error processing {method_name}: {e}")

    # Brute-force all 32-bit seeds with checkpointing
    print("\nBrute-forcing all 32-bit seeds (0 to 2^32-1) on CPU with checkpointing...")
    total_seeds = 2**32
    num_cores = mp.cpu_count()
    chunk_size = total_seeds // num_cores
    seed_ranges = [(i * chunk_size, (i + 1) * chunk_size) for i in range(num_cores)]
    seed_ranges[-1] = (seed_ranges[-1][0], total_seeds)  # Adjust last chunk
    checkpoint_file = "weak_rng_checkpoint.txt"

    print(f"Using {num_cores} cores, each processing ~{chunk_size} seeds...")
    start_time = time()
    results["Weak RNG"] = {"min": float('inf'), "mean": float('inf'), "x_below_ratio": 0, "y_above_ratio": 0, "distances": []}
    top_new_candidates["Weak RNG"] = []

    with mp.Pool(num_cores) as pool:
        brute_results = pool.starmap(test_seed_range, [(r, r_values_ref, f"{checkpoint_file}_{i}") for i, r in enumerate(seed_ranges)])

        for found_key, distance, candidates, distances, x_below_count, y_above_count in brute_results:
            if found_key:
                print("Key found by brute-force, exiting...")
                exit(0)
            top_new_candidates["Weak RNG"].extend(candidates)
            results["Weak RNG"]["distances"].extend(distances)
            results["Weak RNG"]["x_below_count"] = results["Weak RNG"].get("x_below_count", 0) + x_below_count
            results["Weak RNG"]["y_above_count"] = results["Weak RNG"].get("y_above_count", 0) + y_above_count

        top_new_candidates["Weak RNG"].sort(key=lambda x: x["distance"])
        top_new_candidates["Weak RNG"] = top_new_candidates["Weak RNG"][:200]

        min_distance = min(results["Weak RNG"]["distances"]) if results["Weak RNG"]["distances"] else float('inf')
        mean_distance = sum(results["Weak RNG"]["distances"]) / len(results["Weak RNG"]["distances"]) if results["Weak RNG"]["distances"] else float('inf')
        x_below_ratio = results["Weak RNG"]["x_below_count"] / total_seeds
        y_above_ratio = results["Weak RNG"]["y_above_count"] / total_seeds

        results["Weak RNG"]["min"] = min_distance
        results["Weak RNG"]["mean"] = mean_distance
        results["Weak RNG"]["x_below_ratio"] = x_below_ratio
        results["Weak RNG"]["y_above_ratio"] = y_above_ratio

    print(f"\nMethod: Weak RNG (Brute-Force)")
    print(f"Minimum Hamming Distance: {min_distance}")
    print(f"Mean Hamming Distance: {mean_distance:.2f}")
    print(f"Ratio x < target_x: {x_below_ratio:.3f}")
    print(f"Ratio y > target_y: {y_above_ratio:.3f}")
    print(f"Time taken: {(time() - start_time) / 3600:.2f} hours")

    # Save top candidates
    for method_name, candidates in top_new_candidates.items():
        with open(f"{method_name}_top_candidates.txt", "w") as f:
            for cand in candidates:
                if method_name == "Weak RNG":
                    f.write(f"Seed: {cand['seed']}, Private Key: {cand['private_key']}, Distance: {cand['distance']}, "
                            f"x_below: {cand['x_below']}, y_above: {cand['y_above']}, x_match: {cand['x_match']}, "
                            f"y_match: {cand['y_match']}, r_value: {cand['r_value']}\n")
                else:
                    f.write(f"Index: {cand['index']}, Private Key: {cand['private_key']}, Distance: {cand['distance']}, "
                            f"x_below: {cand['x_below']}, y_above: {cand['y_above']}, x_match: {cand['x_match']}, "
                            f"y_match: {cand['y_match']}, r_value: {cand['r_value']}\n")

    # Save distances
    for method_name, stats in results.items():
        with open(f"{method_name}_distances.txt", "w") as f:
            f.write("\n".join(map(str, stats["distances"])))

Starting optimized private key search with CPU brute-force and checkpointing...

Hill-climbing for candidates...
Weak RNG Index 6670011: Original Distance: 195, New Distance: 207, Range: [105629508249606899476590693876604726588334301423823887601494816544247595810893, 105629508249606899476590693876604726588334301423823887601494816544247730028621], Events: [(73807199162307698300053892961764211480390621900966263994699993228588608143272, 'x_cross_to_above', 270, 108204485669484531829675379796499489239244131436127664624015707192206166107813, 72142831652154122806448763014937097481068047787415988080775344385867847789868), (73807199162307698300053892961764211480390621900966263994699993228588608143271, 'y_cross_to_below', 239, 4030578060484488326498678502000593879735737347625931383213073366442785529169, 10886553004852554644240262611179037482361253701831637643491904453139163364331), (31822309087299201176536800914840515107943679522857623606794823315659054792209, 'y_cross_to_above', 259, 355886973

KeyboardInterrupt: 

In [ ]:
# Binary Bit-Locking Cryptographic Attack with Hash160 Hamming Distance
# Install required packages
!pip install -q coincurve numpy base58 pycryptodome

import hashlib
import os
import json
import logging
import time
import sys
import base58
from Crypto.Hash import RIPEMD160
import coincurve
import argparse
import itertools

# Detect Jupyter/Colab environment
def is_jupyter():
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except:
        return False

# Set up logging
logging.basicConfig(filename='binary_debug_log.txt', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# secp256k1 parameters
p = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F
n = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141
G = (
    0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798,
    0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8
)
a, b = 0, 7

# Binary bits and connectors
binary_bits = ['0', '1']
connectors = ['.', '\\', '/', '-', '_', '']

# Parse command-line arguments
def parse_args():
    parser = argparse.ArgumentParser(description="Binary Bit-Locking Cryptographic Attack")
    parser.add_argument("--address-file", type=str, default="addresses.txt", help="Path to text file with P2PKH addresses")
    parser.add_argument("--verbose", action="store_true", help="Print verbose output for debugging")

    if is_jupyter():
        args = parser.parse_args([])
        args.verbose = True
        return args
    else:
        filtered_argv = [arg for arg in sys.argv if not arg.startswith('-f')]
        return parser.parse_args(filtered_argv[1:])

args = parse_args()

# Modular arithmetic
def mod_inverse(a, m):
    def extended_gcd(a, b):
        if a == 0:
            return b, 0, 1
        gcd, x1, y1 = extended_gcd(b % a, a)
        x = y1 - (b // a) * x1
        y = x1
        return gcd, x, y

    _, x, _ = extended_gcd(a, m)
    return (x % m + m) % m

# secp256k1 point operations
def point_add(P, Q):
    if P is None:
        return Q
    if Q is None:
        return P
    x1, y1 = P
    x2, y2 = Q
    if x1 == x2 and y1 != y2:
        return None

    if P == Q:
        lam = ((3 * x1 * x1 + a) * mod_inverse(2 * y1, p)) % p
    else:
        lam = ((y2 - y1) * mod_inverse(x2 - x1, p)) % p

    x3 = (lam * lam - x1 - x2) % p
    y3 = (lam * (x1 - x3) - y1) % p
    return (x3, y3)

def scalar_mult(k, point):
    k = k % n
    result = None
    temp = point
    while k:
        if k & 1:
            result = point_add(result, temp)
        temp = point_add(temp, temp)
        k >>= 1
    return result

# Compute public key from private key
def get_public_key(private_key):
    try:
        private_key_int = int.from_bytes(private_key, 'big')
        if private_key_int < 1 or private_key_int >= n:
            return None, None, None
        point = scalar_mult(private_key_int, G)
        if point is None:
            return None, None, None
        x, y = point
        pubkey = b'\x04' + x.to_bytes(32, 'big') + y.to_bytes(32, 'big')
        return pubkey, x, y
    except Exception as e:
        logging.error(f"Error in get_public_key: {e}")
        return None, None, None

# Hamming distance
def hamming_distance(bytes1, bytes2):
    if len(bytes1) != len(bytes2):
        raise ValueError(f"Byte strings must have equal length")
    return sum(bin(b1 ^ b2).count('1') for b1, b2 in zip(bytes1, bytes2))

# Decode P2PKH address to Hash160
def decode_p2pkh_address(address):
    try:
        decoded = base58.b58decode(address)
        if len(decoded) != 25:
            raise ValueError("Invalid address length")
        version = decoded[0]
        hash160 = decoded[1:21]
        checksum = decoded[21:]
        computed_checksum = hashlib.sha256(hashlib.sha256(decoded[:21]).digest()).digest()[:4]
        if checksum != computed_checksum:
            raise ValueError("Invalid checksum")
        if version != 0x00:  # Mainnet P2PKH
            raise ValueError("Not a mainnet P2PKH address")
        return hash160
    except Exception as e:
        logging.error(f"Error decoding address {address}: {e}")
        return None

# Compute Hash160 from public key
def public_key_to_hash160(pubkey):
    try:
        sha256_hash = hashlib.sha256(pubkey).digest()
        ripemd160 = RIPEMD160.new()
        ripemd160.update(sha256_hash)
        return ripemd160.digest()
    except Exception as e:
        logging.error(f"Error computing Hash160: {e}")
        return None

# Compute Bitcoin address from public key
def public_key_to_bitcoin_address(pubkey, version_byte=b'\x00'):
    try:
        sha256_hash = hashlib.sha256(pubkey).digest()
        ripemd160 = RIPEMD160.new()
        ripemd160.update(sha256_hash)
        hashed_pubkey = ripemd160.digest()
        version_hashed = version_byte + hashed_pubkey
        checksum = hashlib.sha256(hashlib.sha256(version_hashed).digest()).digest()[:4]
        final_bytes = version_hashed + checksum
        address = base58.b58encode(final_bytes).decode('utf-8')
        return address
    except Exception as e:
        logging.error(f"Error computing Bitcoin address: {e}")
        return None

# Compute WIF from private key
def private_key_to_wif(private_key, version_byte=b'\x80'):
    try:
        extended_key = version_byte + private_key
        checksum = hashlib.sha256(hashlib.sha256(extended_key).digest()).digest()[:4]
        final_key = extended_key + checksum
        wif = base58.b58encode(final_key).decode('utf-8')
        return wif
    except Exception as e:
        logging.error(f"Error computing WIF: {e}")
        return None

# Load addresses from file
def load_addresses(file_path):
    try:
        with open(file_path, 'r') as f:
            addresses = [line.strip() for line in f if line.strip()]
        if not addresses:
            raise ValueError("Address file is empty")
        address_hash160s = []
        for addr in addresses:
            hash160 = decode_p2pkh_address(addr)
            if hash160:
                address_hash160s.append((addr, hash160))
        return address_hash160s
    except Exception as e:
        print(f"Error loading address file: {e}")
        print("Please upload the file to Colab or specify the correct path.")
        sys.exit(1)

if is_jupyter():
    address_file = input("Enter the path to the address file (default: addresses.txt): ") or args.address_file
else:
    address_file = args.address_file

TARGET_ADDRESSES = load_addresses(address_file)

# Binary bit-locking generator
def binary_bit_locking_generator(target_address, target_hash160, state):
    locked_prefix = state.get("locked_prefix", [])
    pos = state.get("pos", 1)
    best_distance = state.get("best_distance", float('inf'))
    tried_initial_chars = state.get("tried_initial_chars", [])

    while True:
        # Test all bit and connector combinations
        best_bit = None
        best_connector = None
        best_distance_local = float('inf')
        best_x, best_y = None, None
        best_seed = None
        best_priv = None
        best_pub = None
        best_hash160 = None
        best_address = None

        for bit in binary_bits:
            if not locked_prefix and (bit == '0' or bit in tried_initial_chars):
                continue
            for connector in connectors:
                seed = locked_prefix + [bit]
                if connector:
                    seed.append(connector)
                current_seed = ''.join(seed)

                logging.info(f"Target: {target_address}, Testing seed: {current_seed}, Position: {pos}, Bit: {bit}, Connector: {connector or 'None'}")

                current_priv = hashlib.sha256(current_seed.encode('utf-8')).digest()
                current_pub, x, y = get_public_key(current_priv)
                if current_pub is None:
                    continue

                current_hash160 = public_key_to_hash160(current_pub)
                if current_hash160 is None:
                    continue

                current_distance = hamming_distance(current_hash160, target_hash160)

                if current_distance < best_distance_local:
                    best_distance_local = current_distance
                    best_bit = bit
                    best_connector = connector
                    best_x, best_y = x, y
                    best_seed = current_seed
                    best_priv = current_priv
                    best_pub = current_pub
                    best_hash160 = current_hash160
                    best_address = public_key_to_bitcoin_address(current_pub)

        # Lock the best or least worst bit/connector
        if best_bit is not None:
            locked_prefix.append(best_bit)
            if best_connector:
                locked_prefix.append(best_connector)
            if best_distance_local < best_distance:
                best_distance = best_distance_local
                if best_distance == 0 and best_address == target_address:
                    yield best_seed, best_distance, locked_prefix, pos, best_x, best_y, best_priv, best_pub, best_hash160, best_address, True
                else:
                    yield best_seed, best_distance, locked_prefix, pos, best_x, best_y, best_priv, best_pub, best_hash160, best_address, False
            else:
                yield best_seed, best_distance, locked_prefix, pos, best_x, best_y, best_priv, best_pub, best_hash160, best_address, False
        else:
            logging.warning(f"Target: {target_address}, No valid bit/connector combinations to try at position {pos}. Stopping.")
            break

        # Nibble 1-2 times in 2/4/8/16-bit chunks
        binary_indices = [i for i, c in enumerate(locked_prefix) if c in binary_bits]
        nibble_sizes = []
        if len(binary_indices) >= 2:
            nibble_sizes.append(2)
        if len(binary_indices) >= 4:
            nibble_sizes.append(4)
        if len(binary_indices) >= 8:
            nibble_sizes.append(8)
        if len(binary_indices) >= 16:
            nibble_sizes.append(16)

        nibble_count = 0
        for _ in range(2):
            if nibble_count >= 2:
                break
            improved = False
            for size in nibble_sizes:
                nibble_indices = binary_indices[-size:] if len(binary_indices) >= size else binary_indices
                if len(nibble_indices) < size:
                    continue
                for flip_indices in itertools.combinations(nibble_indices, size):
                    test_prefix = locked_prefix.copy()
                    for i in flip_indices:
                        test_prefix[i] = '1' if test_prefix[i] == '0' else '0'
                    current_seed = ''.join(test_prefix)

                    logging.info(f"Target: {target_address}, Nibbling: Testing seed: {current_seed}, Flipped {size} bits at indices {flip_indices}")

                    current_priv = hashlib.sha256(current_seed.encode('utf-8')).digest()
                    current_pub, x, y = get_public_key(current_priv)
                    if current_pub is None:
                        continue

                    current_hash160 = public_key_to_hash160(current_pub)
                    if current_hash160 is None:
                        continue

                    current_distance = hamming_distance(current_hash160, target_hash160)

                    if current_distance < best_distance:
                        locked_prefix = test_prefix
                        best_distance = current_distance
                        best_x, best_y = x, y
                        best_seed = current_seed
                        best_priv = current_priv
                        best_pub = current_pub
                        best_hash160 = current_hash160
                        best_address = public_key_to_bitcoin_address(current_pub)
                        nibble_count += 1
                        improved = True
                        if best_distance == 0 and best_address == target_address:
                            yield best_seed, best_distance, locked_prefix, pos, best_x, best_y, best_priv, best_pub, best_hash160, best_address, True
                            return
                        else:
                            yield best_seed, best_distance, locked_prefix, pos, best_x, best_y, best_priv, best_pub, best_hash160, best_address, False
                        break
                if improved:
                    break
            if improved:
                continue
            break

        pos += 1

def main():
    # Verify secp256k1 implementation
    test_priv = bytes.fromhex("0000000000000000000000000000000000000000000000000000000000000001")
    pubkey, _, _ = get_public_key(test_priv)
    assert pubkey.hex() == "0479be667ef9dcbbac55a06295ce870b07029bfcdb2dce28d959f2815b16f81798483ada7726a3c4655da4fbfc0e1108a8fd17b448a68554199c47d08ffb10d4b8", "secp256k1 implementation incorrect"

    # Initialize state for each address
    checkpoint_file = "binary_checkpoint.json"
    states = []
    if os.path.exists(checkpoint_file):
        try:
            with open(checkpoint_file, 'r') as f:
                checkpoint_data = json.load(f)
                checkpoint_states = checkpoint_data.get("states", [])
                for target_addr, target_hash160 in TARGET_ADDRESSES:
                    state = next((s for s in checkpoint_states if s["target_address"] == target_addr), None)
                    if state:
                        states.append({
                            "target_address": target_addr,
                            "target_hash160": bytes.fromhex(state["target_hash160"]) if state.get("target_hash160") else target_hash160,
                            "locked_prefix": state.get("locked_prefix", []),
                            "pos": state.get("pos", 1),
                            "best_distance": state.get("best_distance", float('inf')),
                            "best_seed": state.get("best_seed", ""),
                            "best_private_key": bytes.fromhex(state.get("best_private_key", "")) if state.get("best_private_key") else None,
                            "best_public_key": bytes.fromhex(state.get("best_public_key", "")) if state.get("best_public_key") else None,
                            "best_hash160": bytes.fromhex(state.get("best_hash160", "")) if state.get("best_hash160") else None,
                            "best_address": state.get("best_address", None),
                            "tried_initial_chars": state.get("tried_initial_chars", []),
                            "candidates": state.get("candidates", [])
                        })
                    else:
                        states.append({
                            "target_address": target_addr,
                            "target_hash160": target_hash160,
                            "locked_prefix": [],
                            "pos": 1,
                            "best_distance": float('inf'),
                            "best_seed": "",
                            "best_private_key": None,
                            "best_public_key": None,
                            "best_hash160": None,
                            "best_address": None,
                            "tried_initial_chars": [],
                            "candidates": []
                        })
        except Exception as e:
            logging.error(f"Error loading checkpoint: {e}")
            states = [{
                "target_address": addr,
                "target_hash160": hash160,
                "locked_prefix": [],
                "pos": 1,
                "best_distance": float('inf'),
                "best_seed": "",
                "best_private_key": None,
                "best_public_key": None,
                "best_hash160": None,
                "best_address": None,
                "tried_initial_chars": [],
                "candidates": []
            } for addr, hash160 in TARGET_ADDRESSES]
    else:
        states = [{
            "target_address": addr,
            "target_hash160": hash160,
            "locked_prefix": [],
            "pos": 1,
            "best_distance": float('inf'),
            "best_seed": "",
            "best_private_key": None,
            "best_public_key": None,
            "best_hash160": None,
            "best_address": None,
            "tried_initial_chars": [],
            "candidates": []
        } for addr, hash160 in TARGET_ADDRESSES]

    start_time = time.time()
    seeds_processed = 0
    address_index = 0

    while True:
        state = states[address_index]
        target_address = state["target_address"]
        target_hash160 = state["target_hash160"]
        generator = binary_bit_locking_generator(target_address, target_hash160, state)

        try:
            seed, gen_best_distance, locked_prefix, pos, x, y, priv_key, pub_key, hash160, address, is_new_best = next(generator)
            seeds_processed += 1

            state["locked_prefix"] = locked_prefix
            state["pos"] = pos

            if x is not None and y is not None and priv_key is not None and pub_key is not None and address is not None and hash160 is not None:
                if gen_best_distance == 0 and address == target_address:
                    wif = private_key_to_wif(priv_key) if priv_key else None
                    print(f"Address match: Seed={seed}, Distance=0, Private Key={priv_key.hex()}, Address={address}, WIF={wif}, Target Address={target_address}")
                    state["best_distance"] = 0
                    state["best_seed"] = seed
                    state["best_private_key"] = priv_key
                    state["best_public_key"] = pub_key
                    state["best_hash160"] = hash160
                    state["best_address"] = address
                    state["candidates"].append({
                        "seed": seed,
                        "private_key": priv_key.hex(),
                        "public_key": pub_key.hex(),
                        "hash160": hash160.hex(),
                        "distance": 0,
                        "address": address,
                        "x": x,
                        "y": y,
                        "x_below": False,
                        "y_above": False,
                        "x_match": False,
                        "y_match": False
                    })
                    break

                if is_new_best:
                    state["best_distance"] = gen_best_distance
                    state["best_seed"] = seed
                    state["best_private_key"] = priv_key
                    state["best_public_key"] = pub_key
                    state["best_hash160"] = hash160
                    state["best_address"] = address
                    state["candidates"].append({
                        "seed": seed,
                        "private_key": priv_key.hex(),
                        "public_key": pub_key.hex(),
                        "hash160": hash160.hex(),
                        "distance": gen_best_distance,
                        "address": address,
                        "x": x,
                        "y": y,
                        "x_below": False,
                        "y_above": False,
                        "x_match": False,
                        "y_match": False
                    })
                    state["candidates"].sort(key=lambda x: x["distance"])
                    state["candidates"] = state["candidates"][:100]
        except StopIteration:
            logging.warning(f"Target address {target_address} exhausted all possibilities.")
            states[address_index]["exhausted"] = True
            if all(state.get("exhausted", False) for state in states):
                break

        if seeds_processed % 1000 == 0:
            try:
                # Convert bytes to hex for JSON serialization
                serializable_states = []
                for state in states:
                    serializable_state = state.copy()
                    serializable_state["target_hash160"] = state["target_hash160"].hex() if state["target_hash160"] else None
                    serializable_state["best_private_key"] = state["best_private_key"].hex() if state["best_private_key"] else None
                    serializable_state["best_public_key"] = state["best_public_key"].hex() if state["best_public_key"] else None
                    serializable_state["best_hash160"] = state["best_hash160"].hex() if state["best_hash160"] else None
                    serializable_state["candidates"] = [
                        {
                            "seed": cand["seed"],
                            "private_key": cand["private_key"],
                            "public_key": cand["public_key"],
                            "hash160": cand["hash160"],
                            "distance": cand["distance"],
                            "address": cand["address"],
                            "x": cand["x"],
                            "y": cand["y"],
                            "x_below": cand["x_below"],
                            "y_above": cand["y_above"],
                            "x_match": cand["x_match"],
                            "y_match": cand["y_match"]
                        } for cand in state["candidates"]
                    ]
                    serializable_states.append(serializable_state)

                with open(checkpoint_file, 'w', encoding='utf-8') as f:
                    json.dump({
                        "states": serializable_states,
                        "seeds_processed": seeds_processed,
                        "last_update": time.strftime("%Y-%m-%d %H:%M:%S")
                    }, f, ensure_ascii=False)
            except Exception as e:
                logging.error(f"Error saving checkpoint: {e}")

        address_index = (address_index + 1) % len(TARGET_ADDRESSES)

    try:
        with open("binary_top_candidates.txt", "w") as f:
            for state in states:
                for cand in state["candidates"]:
                    f.write(f"Target Address: {state['target_address']}, Seed: {cand['seed']}, "
                            f"Private Key: {cand['private_key']}, Distance: {cand['distance']}, "
                            f"Hash160: {cand['hash160']}, Address: {cand['address']}, "
                            f"WIF: {private_key_to_wif(bytes.fromhex(cand['private_key']))}, "
                            f"x_below: {cand['x_below']}, y_above: {cand['y_above']}, "
                            f"x_match: {cand['x_match']}, y_match: {cand['y_match']}\n")
    except Exception as e:
        logging.error(f"Error saving binary_top_candidates.txt: {e}")

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nSearch stopped by user. Progress saved in checkpoint file.")
    except Exception as e:
        print(f"\nError occurred: {e}")
        print("Progress saved in checkpoint file.")

Enter the path to the address file (default: addresses.txt): addresses.txt
